In [8]:
import torch
import torch.nn as nn
from transformers import AutoModel
from transformers import AutoTokenizer
from transformers import T5ForConditionalGeneration
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import json

class DuReaderQG(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt') as f:
            for idx, line in enumerate(f):
                sample = json.loads(line.strip())
                Data[idx] = sample
        return Data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]
    
train_data = DuReaderQG('DuReaderQG/train.json')
valid_data = DuReaderQG('DuReaderQG/dev.json')

print(train_data[0])

{'context': '第35集雪见缓缓张开眼睛，景天又惊又喜之际，长卿和紫萱的仙船驶至，见众人无恙，也十分高兴。众人登船，用尽合力把自身的真气和水分输给她。雪见终于醒过来了，但却一脸木然，全无反应。众人向常胤求助，却发现人世界竟没有雪见的身世纪录。长卿询问清微的身世，清微语带双关说一切上了天界便有答案。长卿驾驶仙船，众人决定立马动身，往天界而去。众人来到一荒山，长卿指出，魔界和天界相连。由魔界进入通过神魔之井，便可登天。众人至魔界入口，仿若一黑色的蝙蝠洞，但始终无法进入。后来花楹发现只要有翅膀便能飞入。于是景天等人打下许多乌鸦，模仿重楼的翅膀，制作数对翅膀状巨物。刚佩戴在身，便被吸入洞口。众人摔落在地，抬头发现魔界守卫。景天和众魔套交情，自称和魔尊重楼相熟，众魔不理，打了起来。', 'answer': '第35集', 'question': '仙剑奇侠传3第几集上天界', 'id': 0}


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import T5ForConditionalGeneration
from transformers import AutoTokenizer
import transformers.modeling_utils as modeling_utils
import transformers.utils.import_utils as import_utils

checkpoint = "./mengzi-t5-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True)

max_input_length = 512
max_target_length = 64

device = 'cuda' if torch.cuda.is_available() else 'cpu'

modeling_utils.check_torch_load_is_safe = lambda: None
import_utils.check_torch_load_is_safe = lambda: None

model = T5ForConditionalGeneration.from_pretrained(checkpoint, local_files_only=True)
model = model.to(device)

def collote_fn(batch_samples):
    batch_inputs, batch_targets = [], []
    for sample in batch_samples:
        input_text = f"问题：{sample['question']} 上下文：{sample['context']}"
        batch_inputs.append(input_text)
        batch_targets.append(sample['answer'])
    batch_data = tokenizer(
        batch_inputs,
        padding = True,
        max_length = max_input_length,
        truncation = True,
        return_tensors = 'pt'
    )

    labels = tokenizer(
        batch_targets,
        padding = True,
        max_length = max_target_length,
        truncation=True,
        return_tensors="pt"
    )["input_ids"] 
    labels[labels == tokenizer.pad_token_id] = -100
    batch_data['labels'] = labels
    
        
    return batch_data

train_dataloader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=collote_fn)
valid_dataloader = DataLoader(valid_data, batch_size=4, shuffle=False, collate_fn=collote_fn)

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 9304.88it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [10]:
batch = next(iter(train_dataloader))
print(batch.keys())
print('batch shape:', {k: v.shape for k, v in batch.items()})
print(batch)

KeysView({'input_ids': tensor([[    7,   143,    13, 11234,   249,   998, 10745,   270,  2062,  1242,
          3653,    98,     7,  2868,   180,    13,   513,  1707,  1410,     5,
          1586,    85, 23596,     3,   270,  2062,  1242,     5,  3917,   466,
          6493,  1247,     3,     8,   201,   132,  2104,  4829,   431, 12672,
          1762,   392,   270,  2062,  1242,    56,  4829,     3,    72,  1465,
            11,  1354,   316,  4962,     5,     3,   270,  2062,  1242,    63,
          6005,   316,  4962,     5, 23817,     3,   380,   316,  4962,    11,
         10745,  5126, 23817,  1465,     3,    68,    87,  2499,     5,   152,
           467,  1953,     5,   223,   132,    81,   200,     3,   450,  1025,
           342,  1465,   526,   883, 12251,     5,     3,   209,  2336,   270,
          2062,  1242,   234,   839,  3900,  1934,     4, 14980,    45,    98,
            85, 23596,     3,   186, 11960,    66,  5584,  5648,     3,  2029,
            24,  1604,   611,

In [11]:
from tqdm.auto import tqdm

def train_loop(dataloader, model, optimizer, lr_scheduler, epoch):
    total_epoch_loss = 0.0
    progress_bar = tqdm(range(len(dataloader)))
    
    model.train()
    
    for batch, batch_data in enumerate(dataloader, start=1):
        batch_data = batch_data.to(device)
        outputs = model(**batch_data)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_epoch_loss += loss.item()
        progress_bar.set_description(f'Epoch {epoch} Loss: {total_epoch_loss / batch:.7f}')
        progress_bar.update(1)
    return total_epoch_loss / len(dataloader)

In [12]:
import numpy as np
from sacrebleu.metrics import BLEU

bleu_metric = BLEU(effective_order=True)

def test_loop(dataloader, model):
    preds, refs = [], []
    
    model.eval()
    for batch_data in tqdm(dataloader):
        batch_data = batch_data.to(device)
        with torch.no_grad():
            generated_tokens = model.generate(
                batch_data["input_ids"],
                attention_mask=batch_data["attention_mask"],
                max_length=max_target_length,
                num_beams=4,
                no_repeat_ngram_size=2,
            ).cpu().numpy()

        if isinstance(generated_tokens, tuple):
            generated_tokens = generated_tokens[0]
        
        label_tokens = batch_data["labels"].cpu().numpy()
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        label_tokens = np.where(label_tokens != -100, label_tokens, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

        preds.extend(decoded_preds)
        refs.extend(decoded_labels)

    bleu_result = bleu_metric.corpus_score(preds, [refs])
    precisions = list(bleu_result.precisions)
    results = {"bleu": bleu_result.score, "precisions": precisions}

    bleu_1, bleu_2, bleu_3, bleu_4 = precisions
    print(f"bleu_1:{bleu_1}; bleu_2:{bleu_2}; bleu_3:{bleu_3}; bleu_4:{bleu_4}")

    return results


Using the latest cached version of the module from C:\Users\jd\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--bleu\9e0985c1200e367cce45605ce0ecb5ede079894e0f24f54613fca08eeb8aff76 (last modified on Sun Mar 22 11:33:27 2026) since it couldn't be found locally at evaluate-metric--bleu, or remotely on the Hugging Face Hub.


In [ ]:
from transformers import get_scheduler
from torch.optim import AdamW
import matplotlib.pyplot as plt

learning_rate = 2e-5
epoch_num = 2

optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=epoch_num*len(train_dataloader),
)

history = {'train_loss': [], 'valid_bleu': []}
best_bleu_avg = 0.
output_dir = "./saved_model" # 模型保存路径
for t in range(epoch_num):
    epoch = t + 1
    print(f"Epoch {epoch}/{epoch_num}\n-------------------------------")
    
    # 返回当前 epoch 的平均 loss
    avg_train_loss = train_loop(train_dataloader, model, optimizer, lr_scheduler, epoch)
    print(f"Epoch {epoch} - Average Train Loss: {avg_train_loss:.4f}")
    
    # 将当前 epoch 的平均 loss 存入 history
    history['train_loss'].append(avg_train_loss)
    
    valid_results = test_loop(valid_dataloader, model)
    print(f"Epoch {epoch} - Validation Results: {valid_results}")
    
    # 将验证集 bleu 分数存入 history
    history['valid_bleu'].append(valid_results['bleu'])
    # 保存最佳模型
    if valid_results['bleu'] > best_bleu_avg:
        best_bleu_avg = valid_results['bleu']
        print(f"Best BLEU score: {best_bleu_avg:.4f}. Saving model to {output_dir}")
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
print("Done")


plt.figure(figsize=(12, 5))
# 绘制训练损失曲线
plt.subplot(1, 2, 1)
plt.plot(range(1, epoch_num + 1), history['train_loss'], label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()
# 绘制验证集BLEU分数曲线
plt.subplot(1, 2, 2)
plt.plot(range(1, epoch_num + 1), history['valid_bleu'], label='Validation BLEU', color='orange')
plt.xlabel('Epoch')
plt.ylabel('BLEU Score')
plt.title('Validation BLEU Score Curve')
plt.legend()
plt.tight_layout()
plt.show()

Epoch 1/2
-------------------------------


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
